## Reinforcement Learning Notes

Two core components that will be involved here: **value learning** and **action selection**.

### Value Learning (Rescorla–Wagner Update)

On each trial, the agent maintains an estimate $V_i^{(t)}$ of how likely stimulus $i$ is to produce the aversive sound. After choosing stimulus $i$ and observing outcome $o^{(t)} \in \{0,1\}$, the value is updated as:

$$
V_i^{(t+1)} = V_i^{(t)} + \alpha \left(o^{(t)} - V_i^{(t)}\right)
$$

where:

- $o^{(t)} = 1$ indicates an aversive sound
- $o^{(t)} = 0$ indicates no sound
- $\alpha \in (0,1)$ is the learning rate

The term

$$
\delta^{(t)} = o^{(t)} - V_i^{(t)}
$$

is the prediction error, which is the discrepancy between the observed outcome and the current expectation.

This means:

- if the outcome is worse than expected, the value increases
- if the outcome is safer than expected, the value decreases
- only the chosen stimulus is updated on that trial

In this task, a higher value means that the option is currently estimated to be more likely to produce the aversive sound.

### Action Selection (Softmax)

Given the current values of the two options, the probability of choosing stimulus A is:

$$
P(\text{choose A}) =
\frac{\exp(-\beta V_A)}
{\exp(-\beta V_A) + \exp(-\beta V_B)}
$$

where $\beta$ is the inverse temperature parameter.

The negative sign is important because this is an avoidance-learning task. Larger values correspond to more aversive options.


- when $\beta$ is close to 0, choices become more random
- when $|\beta|$ is large, choices become more deterministic
- in this avoidance setting, the choice rule biases behaviour toward the option with lower expected aversiveness

## Use a generative model to simulate data

Model 1 is used as a generative model to simulate artificial behavioural data from known parameter values. This lets us verify our code, explore model behaviour, and later validate our fitting pipeline.

In [2]:
def simulate_model1(alpha, beta, n_trials=N_TRIALS, v0=V0, return_values=False):
    Va, Vb = v0, v0
    ch, ou = [], []
    Va_h, Vb_h = [Va], [Vb]

    for t in range(n_trials):
        # Softmax
        p_a = np.exp(-beta * Va) / (np.exp(-beta * Va) + np.exp(-beta * Vb))
        choice = 1 if np.random.rand() < p_a else 0

        # Generate outcome from block probabilities
        block = min(t // BLOCK_SIZE, 3)
        p_aversive = PROB_A[block] if choice == 1 else (1 - PROB_A[block])
        outcome = 1 if np.random.rand() < p_aversive else 0

        # Update the chosen stimulus
        if choice == 1:
            Va += alpha * (outcome - Va)
        else:
            Vb += alpha * (outcome - Vb)

        ch.append(choice); ou.append(outcome)
        Va_h.append(Va); Vb_h.append(Vb)

    if return_values:
        return np.array(ch), np.array(ou), np.array(Va_h), np.array(Vb_h)
    return np.array(ch), np.array(ou)

print("simulate_model1() defined")

NameError: name 'N_TRIALS' is not defined

### Running many simulations and averaging
As the model is stochastic, a single run is noisy. We average over many simulations to see the model's expected behaviour.

In [ ]:
alpha, beta = 0.3, -8
n_sims = 500

all_Va, all_Vb, all_sounds = [], [], []
for _ in range(n_sims):
    _, o, Va_h, Vb_h = simulate_model1(alpha, beta, return_values=True)
    all_Va.append(Va_h); all_Vb.append(Vb_h); all_sounds.append(o.sum())

mean_Va = np.mean(all_Va, axis=0)
mean_Vb = np.mean(all_Vb, axis=0)
print(f"Mean aversive sounds over {n_sims} sims: {np.mean(all_sounds):.2f} (hint: ~60)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
trials = np.arange(N_TRIALS + 1)

axes[0].plot(trials, mean_Va, label="V(A)", color="tab:red")
axes[0].plot(trials, mean_Vb, label="V(B)", color="tab:blue")
for b in range(1, 4): axes[0].axvline(b*BLOCK_SIZE, color="grey", ls="--", lw=1, alpha=0.6)
axes[0].set_xlabel("Trial"); axes[0].set_ylabel("Value estimate")
axes[0].set_title(f"Value trajectories (α={alpha}, β={beta})"); axes[0].legend()

diff = mean_Va - mean_Vb
axes[1].plot(trials, diff, color="tab:purple")
axes[1].axhline(0, color="grey", ls=":", lw=1)
for b in range(1, 4): axes[1].axvline(b*BLOCK_SIZE, color="grey", ls="--", lw=1, alpha=0.6)
axes[1].set_xlabel("Trial"); axes[1].set_ylabel("V(A) − V(B)")
axes[1].set_title("Value difference")

plt.tight_layout(); plt.savefig("figures/task_b.png"); plt.show()

In [ ]:
**Interpretation:** V(A) > V(B) because A leads to sound more often. The difference is largest in block 2 (80/20) and smallest in block 4 (65/35). At block boundaries, values adjust as the agent tracks the new contingencies.